In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH= "telecom_guide.pdf"
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the pdf")
print("\n -- first page preview(first 500 chars)---")
print(pages[0].page_content[:500])

/var/folders/62/b_v8vpgn355932wzt8r9j_qr0000gn/T/ipykernel_24991/2742290077.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/biruk/Documents/Projects/GenAI/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9 pages from the pdf

 -- first page preview(first 500 chars)---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 100,
    separators = ["\n\n","\n","."," "],
)

chunks = splitter.split_documents(pages)
len(chunks)

37

In [4]:
chunks[0].page_content


'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks,embeddings)

print(f"Vectore store value is {vector_store._collection.count()}")

Loading weights: 100%|████| 103/103 [00:00<00:00, 7917.70it/s]


Vectore store value is 37


In [6]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

test_query = "What is voLTE and hoe does it improve call quality?"
retrived = retriever.invoke(test_query)

for i, doc in enumerate(retrived,1):
        print(doc.page_content[:300])

Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call cannot be established  - for exam


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

def format_docs(docs):
        return "\n\n---\n\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT = """\
you are a helpful telecom assistant.
Answer the question using only the context provided below.
If the context does not contain enough information,say no clearly.

context:
{context}


"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human","{question}"),
])

llm= ChatGroq(
    model = "qwen/qwen3.8-27b",
    temperature = 0,
    reasoning_format = "parsed",
    
)

chain = ({"context":retriever|format_docs,"question":RunnablePassthrough()}
        |prompt
        |llm
        |StrOutputParser()
        
        )

print("RAG chain assembeled.")

RAG chain assembeled.


In [10]:
question = "How does international roaming work and what charges should i expext?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges should i expext?

A: Based on the provided context, here is how international roaming works and the expected charges:

**How It Works:**
*   **Connection:** When you travel outside your home network's coverage, your device connects to a partner network in the visited country.
*   **Authentication & Billing:** The visited network authenticates you via signalling protocols (SS7 or Diameter). Your home network validates your subscription and authorizes service. All voice, data, and SMS traffic is tunnelled back to the home network for billing.
*   **Activation:** Roaming must be enabled on your account **before** departure. You can do this via the MyTelecom app (Plan & Services > International Roaming) or by calling 611. Network-level activation takes up to 15 minutes. If enabled after landing, you may experience a brief period of no service while the HLR record updates.
*   **Latency:** Because traffic is tunnelled back to the home 